# VLM PDF Extractor — Colab Demo

Upload a PDF, run the full extraction pipeline, and view structured JSON output.

**Requirements:** Google Colab with T4 GPU runtime.

In [ ]:
# Cell 1: Install dependencies
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes qwen-vl-utils[decord] datasets pillow
!pip install -q PyMuPDF opencv-python-headless gradio

In [ ]:
#If using llama vision model 
from huggingface_hub import login
login()

In [ ]:
# Cell 2: Clone repo (skip if already cloned)
import os
if not os.path.exists("VLM_PDF_Extractor"):
    !git clone https://github.com/RahulReadd/VLM_PDF_Extractor.git
os.chdir("VLM_PDF_Extractor")
!pwd

In [ ]:
# Cell 3: Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 4: Import pipeline modules
import sys, json
sys.path.insert(0, ".")

from app.pdf_to_image import convert_pdf, load_image, print_diagnostics
from app.extract import DocumentExtractor, parse_json_output
from app.evaluate import (
    evaluate_single, parse_cord_ground_truth
)

print("Modules loaded successfully.")

In [ ]:
# Cell 5: Load the VLM (change model name as needed)
MODEL_NAME = "qwen3-vl-2b"  # Options: qwen3-vl-2b, qwen3-vl-4b, internvl35-8b-4bit, etc.

extractor = DocumentExtractor(model_name=MODEL_NAME)
print(f"Model '{MODEL_NAME}' loaded and ready.")

In [ ]:
# Cell 6: Gradio UI
import gradio as gr
from PIL import Image

def process_pdf(pdf_file):
    """Process an uploaded PDF — unified extraction (all 4 aspects)."""
    if pdf_file is None:
        return "No file uploaded.", None

    images, diagnostics = convert_pdf(pdf_file.name)

    diag_text = []
    for d in diagnostics:
        px = d.final_size[0] * d.final_size[1]
        diag_text.append(
            f"Page {d.page_num}: {d.width_in}\"x{d.height_in}\" ({d.page_type}) "
            f"| DPI={d.optimal_dpi} | skew={d.skew_angle} deg "
            f"| {d.final_size[0]}x{d.final_size[1]} ({px/1e6:.1f}M px)"
        )

    all_results = []
    for i, img in enumerate(images):
        result = extractor.extract(img)  # unified prompt, no task
        all_results.append({
            "page": i + 1,
            "json_valid": result["json_valid"],
            "extraction": result["parsed_json"],
        })

    output = {
        "model": MODEL_NAME,
        "mode": "full_extraction",
        "diagnostics": diag_text,
        "pages": all_results,
    }
    return json.dumps(output, indent=2, ensure_ascii=False), images[0] if images else None


def process_image(image):
    """Process an uploaded image — unified extraction."""
    if image is None:
        return "No image uploaded."
    img = Image.fromarray(image).convert("RGB")
    result = extractor.extract(img)  # unified prompt
    return json.dumps(result["parsed_json"], indent=2, ensure_ascii=False)


with gr.Blocks(title="VLM PDF Extractor") as demo:
    gr.Markdown("# VLM PDF Extractor")
    gr.Markdown(f"**Model:** {MODEL_NAME} | **GPU:** T4")
    gr.Markdown("Extracts **key-value pairs**, **signature detection**, **form field status**, and **receipt data** in a single pass.")

    with gr.Tab("PDF Upload"):
        pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
        pdf_btn = gr.Button("Extract All", variant="primary")
        with gr.Row():
            pdf_preview = gr.Image(label="First Page Preview")
            pdf_output = gr.Textbox(label="Extracted JSON", lines=25)
        pdf_btn.click(process_pdf, [pdf_input], [pdf_output, pdf_preview])

    with gr.Tab("Image Upload"):
        img_input = gr.Image(label="Upload Image")
        img_btn = gr.Button("Extract All", variant="primary")
        img_output = gr.Textbox(label="Extracted JSON", lines=25)
        img_btn.click(process_image, [img_input], [img_output])

demo.launch(share=True)